In [1]:
# This is code that generates tracking of each camera

In [2]:
import os
import cv2
import json
import numpy as np
from ultralytics import YOLO
from collections import deque
from datetime import datetime


In [3]:
class RobustTracker:
    def __init__(self, camera_code, max_history=30, max_disappeared=15):
        self.id_label_map = {}
        self.track_history = {}
        self.disappeared_tracks = {}
        self.vehicle_counter = 0
        self.pedestrian_counter = 0
        self.max_history = max_history
        self.max_disappeared = max_disappeared
        self.camera_code = camera_code
        
    def get_camera_prefix(self):
        """Map camera folder names to their corresponding prefix codes"""
        camera_prefixes = {
            'Camera_Back': 'BB',
            'Camera_BackLeft': 'BL',
            'Camera_BackRight': 'BR',
            'Camera_Front': 'FF',
            'Camera_FrontLeft': 'FL',
            'Camera_FrontRight': 'FR'
        }
        return camera_prefixes.get(self.camera_code, 'XX')
    
    def assign_label(self, cls_name, vehicle_classes):
        """Assign a new label for a track with camera-specific prefix"""
        prefix = self.get_camera_prefix()
        
        if cls_name in vehicle_classes:
            letter_id = chr(ord('A') + self.vehicle_counter % 26)
            label = f"Veh_{prefix}_{letter_id}"
            self.vehicle_counter += 1
        else:
            letter_id = chr(ord('A') + self.pedestrian_counter % 26)
            label = f"Ped_{prefix}_{letter_id}"
            self.pedestrian_counter += 1
        return label

    def calculate_features(self, image, box):
        """Enhanced feature extraction using HSV color space and multiple regions"""
        x1, y1, x2, y2 = map(int, box)
        roi = image[y1:y2, x1:x2]
        if roi.size == 0:
            return None

        try:
            # Convert to HSV for better color representation
            hsv_roi = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
            
            # Divide ROI into upper and lower regions
            height = roi.shape[0]
            upper_roi = hsv_roi[:height//2, :]
            lower_roi = hsv_roi[height//2:, :]
            
            # Calculate histograms for each region
            hist_bins = [8, 8, 8]  # H, S, V bins
            hist_ranges = [180, 256, 256]  # H, S, V ranges
            
            features = {}
            
            # Upper region histograms
            for i, channel in enumerate(['h', 's', 'v']):
                hist = cv2.calcHist([upper_roi], [i], None, [hist_bins[i]], [0, hist_ranges[i]])
                hist = cv2.normalize(hist, hist).flatten()
                features[f'upper_{channel}'] = hist.tolist()
            
            # Lower region histograms
            for i, channel in enumerate(['h', 's', 'v']):
                hist = cv2.calcHist([lower_roi], [i], None, [hist_bins[i]], [0, hist_ranges[i]])
                hist = cv2.normalize(hist, hist).flatten()
                features[f'lower_{channel}'] = hist.tolist()
            
            # Calculate dominant colors
            pixels = hsv_roi.reshape(-1, 3)
            pixels = np.float32(pixels)
            
            criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 10, 1.0)
            K = 3  # Number of dominant colors
            _, labels, centers = cv2.kmeans(pixels, K, None, criteria, 10, cv2.KMEANS_RANDOM_CENTERS)
            
            # Convert centers back to uint8 and store with percentages
            centers = np.uint8(centers)
            labels_unique, counts = np.unique(labels, return_counts=True)
            percentages = counts / len(labels)
            
            dominant_colors = []
            for center, percentage in zip(centers, percentages):
                dominant_colors.append({
                    'color': center.tolist(),
                    'percentage': float(percentage)
                })
            
            features['dominant_colors'] = dominant_colors
            
            return features
            
        except Exception as e:
            print(f"Error calculating features: {str(e)}")
            return None
    
    def calculate_similarity_score(self, hist1, hist2):
        """Calculate similarity between two feature sets"""
        if hist1 is None or hist2 is None:
            return 0
            
        try:
            # Compare histograms for each region and channel
            total_score = 0
            weight = 1.0 / 6  # Equal weight for each histogram comparison
            
            for region in ['upper', 'lower']:
                for channel in ['h', 's', 'v']:
                    key = f'{region}_{channel}'
                    score = cv2.compareHist(
                        np.array(hist1[key]).reshape(-1, 1),
                        np.array(hist2[key]).reshape(-1, 1),
                        cv2.HISTCMP_CORREL
                    )
                    total_score += weight * max(0, score)  # Ensure non-negative
            
            # Add dominant color comparison if available
            if 'dominant_colors' in hist1 and 'dominant_colors' in hist2:
                color_sim = self.compare_dominant_colors(
                    hist1['dominant_colors'],
                    hist2['dominant_colors']
                )
                total_score = 0.7 * total_score + 0.3 * color_sim
            
            return total_score
            
        except Exception as e:
            print(f"Error calculating similarity: {str(e)}")
            return 0
    
    def compare_dominant_colors(self, colors1, colors2):
        """Compare dominant color sets"""
        if not colors1 or not colors2:
            return 0
            
        total_sim = 0
        for c1 in colors1:
            color1 = np.array(c1['color'])
            pct1 = c1['percentage']
            
            max_color_sim = 0
            for c2 in colors2:
                color2 = np.array(c2['color'])
                pct2 = c2['percentage']
                
                # Calculate color similarity
                color_dist = np.exp(-np.sum(np.abs(color1 - color2)) / 255.0)
                # Weight by both percentages
                sim = color_dist * min(pct1, pct2)
                max_color_sim = max(max_color_sim, sim)
            
            total_sim += max_color_sim
            
        return total_sim / len(colors1)

    def predict_next_position(self, positions):
        """Predict next position based on velocity history"""
        if len(positions) < 2:
            return None
            
        recent_positions = list(positions)[-5:]
        if len(recent_positions) < 2:
            return recent_positions[-1]
            
        velocities = []
        for i in range(1, len(recent_positions)):
            dx = recent_positions[i][0] - recent_positions[i-1][0]
            dy = recent_positions[i][1] - recent_positions[i-1][1]
            dt = 1
            velocities.append((dx/dt, dy/dt))
            
        avg_velocity = np.mean(velocities, axis=0)
        last_pos = recent_positions[-1]
        
        predicted_x = last_pos[0] + avg_velocity[0]
        predicted_y = last_pos[1] + avg_velocity[1]
        
        return (predicted_x, predicted_y)
    
    def update(self, image, current_detections, vehicle_classes):
        """Update tracking for current frame"""
        current_track_ids = set()
        
        for track_id, (box, cls_name) in current_detections.items():
            current_track_ids.add(track_id)
            center = ((box[0] + box[2])/2, (box[1] + box[3])/2)
            features = self.calculate_features(image, box)
            
            if track_id not in self.track_history:
                self.track_history[track_id] = {
                    'positions': deque(maxlen=self.max_history),
                    'features': features,
                    'class': cls_name
                }
                
                best_match_id = None
                best_match_score = 0
                
                for lost_id, lost_data in list(self.disappeared_tracks.items()):
                    if lost_data['class'] != cls_name:
                        continue
                        
                    predicted_pos = self.predict_next_position(lost_data['positions'])
                    if predicted_pos is None:
                        continue
                    
                    dist = np.sqrt((center[0] - predicted_pos[0])**2 + 
                                 (center[1] - predicted_pos[1])**2)
                    pos_score = np.exp(-dist / 100)
                    
                    appear_score = self.calculate_similarity_score(features, 
                                                                lost_data['features'])
                    
                    if cls_name == 'person':
                        total_score = 0.7 * appear_score + 0.3 * pos_score
                    else:
                        total_score = 0.6 * appear_score + 0.4 * pos_score
                    
                    if total_score > best_match_score and total_score > 0.4:
                        best_match_score = total_score
                        best_match_id = lost_id
                
                if best_match_id is not None:
                    self.id_label_map[track_id] = self.disappeared_tracks[best_match_id]['label']
                    self.track_history[track_id]['positions'] = self.disappeared_tracks[best_match_id]['positions']
                    del self.disappeared_tracks[best_match_id]
                else:
                    self.id_label_map[track_id] = self.assign_label(cls_name, vehicle_classes)
            
            self.track_history[track_id]['positions'].append(center)
            self.track_history[track_id]['features'] = features
        
        for track_id in list(self.track_history.keys()):
            if track_id not in current_track_ids:
                if track_id not in self.disappeared_tracks:
                    self.disappeared_tracks[track_id] = {
                        'count': 0,
                        'label': self.id_label_map[track_id],
                        'positions': self.track_history[track_id]['positions'],
                        'features': self.track_history[track_id]['features'],
                        'class': self.track_history[track_id]['class']
                    }
                self.disappeared_tracks[track_id]['count'] += 1
                
                if self.disappeared_tracks[track_id]['count'] > self.max_disappeared:
                    del self.disappeared_tracks[track_id]
                    del self.track_history[track_id]
                    del self.id_label_map[track_id]




In [4]:
def process_folder(image_dir, model, vehicle_classes, target_classes, class_names):
    """Process all images in a single folder"""
    output_dir = os.path.join(image_dir, 'Segmented_Data')
    json_dir = os.path.join(image_dir, 'Annotations_JSON')
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(json_dir, exist_ok=True)
    
    # Get camera folder name from path
    camera_folder = os.path.basename(os.path.dirname(image_dir))
    
    # Initialize tracker with camera code
    tracker = RobustTracker(camera_folder)
    color = (0, 255, 0)
    
    filenames = sorted([f for f in os.listdir(image_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
    
    print(f"Processing folder: {image_dir}")
    print(f"Found {len(filenames)} images")
    
    for filename in filenames:
        image_path = os.path.join(image_dir, filename)
        image = cv2.imread(image_path)
        
        if image is None:
            print(f"Failed to read image: {image_path}")
            continue
        
        # Initialize COCO-format JSON structure for this frame
        json_data = {
            "info": {
                "description": "Traffic Scene Detection Results",
                "url": "",
                "version": "1.0",
                "year": datetime.now().year,
                "contributor": "YOLO Tracker",
                "date_created": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            },
            "images": [{
                "id": 1,
                "file_name": filename,
                "width": image.shape[1],
                "height": image.shape[0]
            }],
            "annotations": [],
            "categories": [
                {"id": 1, "name": "vehicle", "supercategory": "traffic"},
                {"id": 2, "name": "pedestrian", "supercategory": "traffic"}
            ]
        }
        
        results = model.track(source=image, persist=True, stream=True, conf=0.4)
        current_detections = {}
        annotation_id = 1
        
        for result in results:
            boxes = result.boxes
            for box in boxes:
                cls_id = int(box.cls.item())
                cls_name = class_names[cls_id]
                track_id = int(box.id.item()) if box.id is not None else None
                confidence = float(box.conf.item())
                
                if cls_name in target_classes and track_id is not None:
                    x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
                    current_detections[track_id] = ([x1, y1, x2, y2], cls_name)
                    
                    # Calculate additional metrics for JSON
                    width = x2 - x1
                    height = y2 - y1
                    area = width * height
                    center_x = x1 + width/2
                    center_y = y1 + height/2
        
        tracker.update(image, current_detections, vehicle_classes)
        
        # Process detections and create annotations
        for track_id, (box, cls_name) in current_detections.items():
            x1, y1, x2, y2 = box
            label = tracker.id_label_map.get(track_id, f"Unknown_{track_id}")
            
            # Draw on image
            cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)
            cv2.putText(
                image,
                label,
                (x1, y1 - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                1.3,
                color,
                2,
            )
            
            # Add to JSON annotations
            width = x2 - x1
            height = y2 - y1
            
            # Extract features for the annotation
            features = tracker.calculate_features(image, box)
            
            annotation = {
                "id": annotation_id,
                "image_id": 1,
                "category_id": 1 if cls_name in vehicle_classes else 2,
                "track_id": label,  # Store the consistent tracking ID
                "bbox": [x1, y1, width, height],  # [x, y, width, height]
                "area": width * height,
                "segmentation": [],  # Empty as we don't have segmentation masks
                "iscrowd": 0,
                "attributes": {
                    "class": cls_name,
                    "confidence": confidence if 'confidence' in locals() else None,
                    "center": [float(x1 + width/2), float(y1 + height/2)],
                    "features": features  # Add the extracted features to JSON
                }
            }
            
            json_data["annotations"].append(annotation)
            annotation_id += 1
        
        # Save image with annotations
        output_path = os.path.join(output_dir, filename)
        cv2.imwrite(output_path, image)
        
        # Save JSON annotations
        json_filename = os.path.splitext(filename)[0] + '.json'
        json_path = os.path.join(json_dir, json_filename)
        
        with open(json_path, 'w') as f:
            json.dump(json_data, f, indent=2)
        
        print(f"Processed and saved: {output_path}")
        print(f"Saved annotations: {json_path}")



In [5]:
def process_all_cameras(root_dir, model, vehicle_classes, target_classes, class_names):
    """Process all camera folders in the root directory"""
    camera_folders = [
        d for d in os.listdir(root_dir) 
        if os.path.isdir(os.path.join(root_dir, d)) and d.startswith('Camera_')
    ]
    
    for camera_folder in camera_folders:
        camera_path = os.path.join(root_dir, camera_folder)
        
        # Get all subdirectories in the camera folder
        subdirs = [
            d for d in os.listdir(camera_path) 
            if os.path.isdir(os.path.join(camera_path, d)) and d != 'Segmented_Data'
        ]
        
        for subdir in subdirs:
            image_dir = os.path.join(camera_path, subdir)
            try:
                process_folder(image_dir, model, vehicle_classes, target_classes, class_names)
            except Exception as e:
                print(f"Error processing folder {subdir} in {camera_folder}: {str(e)}")
                continue



In [6]:
# Initialize YOLO model and classes
model = YOLO('yolo11x.pt')
vehicle_classes = ['bicycle', 'car', 'motorcycle', 'bus', 'train', 'truck', 'boat']
target_classes = vehicle_classes + ['person']
class_names = model.names



In [7]:
# Set root directory to process all camera folders
root_dir = 'type1_subtype1_accident/ego_vehicle/'
process_all_cameras(root_dir, model, vehicle_classes, target_classes, class_names)

Processing folder: type1_subtype1_accident/ego_vehicle/Camera_Back/Town02_type001_subtype0001_scenario00013
Found 45 images

0: 384x640 1 car, 167.7ms
Speed: 8.1ms preprocess, 167.7ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)
Processed and saved: type1_subtype1_accident/ego_vehicle/Camera_Back/Town02_type001_subtype0001_scenario00013/Segmented_Data/Town02_type001_subtype0001_scenario00013_001.jpg
Saved annotations: type1_subtype1_accident/ego_vehicle/Camera_Back/Town02_type001_subtype0001_scenario00013/Annotations_JSON/Town02_type001_subtype0001_scenario00013_001.json

0: 384x640 1 car, 17.9ms
Speed: 3.0ms preprocess, 17.9ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)
Processed and saved: type1_subtype1_accident/ego_vehicle/Camera_Back/Town02_type001_subtype0001_scenario00013/Segmented_Data/Town02_type001_subtype0001_scenario00013_002.jpg
Saved annotations: type1_subtype1_accident/ego_vehicle/Camera_Back/Town02_type001_subtype0001_scenario000

0: 384x640 1 car, 15.2ms
Speed: 2.9ms preprocess, 15.2ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)
Processed and saved: type1_subtype1_accident/ego_vehicle/Camera_Back/Town02_type001_subtype0001_scenario00013/Segmented_Data/Town02_type001_subtype0001_scenario00013_018.jpg
Saved annotations: type1_subtype1_accident/ego_vehicle/Camera_Back/Town02_type001_subtype0001_scenario00013/Annotations_JSON/Town02_type001_subtype0001_scenario00013_018.json

0: 384x640 1 car, 1 truck, 17.7ms
Speed: 2.6ms preprocess, 17.7ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)
Processed and saved: type1_subtype1_accident/ego_vehicle/Camera_Back/Town02_type001_subtype0001_scenario00013/Segmented_Data/Town02_type001_subtype0001_scenario00013_019.jpg
Saved annotations: type1_subtype1_accident/ego_vehicle/Camera_Back/Town02_type001_subtype0001_scenario00013/Annotations_JSON/Town02_type001_subtype0001_scenario00013_019.json

0: 384x640 2 cars, 17.9ms
Speed: 2.5ms preproce


0: 384x640 1 car, 17.2ms
Speed: 2.7ms preprocess, 17.2ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)
Processed and saved: type1_subtype1_accident/ego_vehicle/Camera_Back/Town02_type001_subtype0001_scenario00013/Segmented_Data/Town02_type001_subtype0001_scenario00013_036.jpg
Saved annotations: type1_subtype1_accident/ego_vehicle/Camera_Back/Town02_type001_subtype0001_scenario00013/Annotations_JSON/Town02_type001_subtype0001_scenario00013_036.json

0: 384x640 1 car, 16.2ms
Speed: 2.5ms preprocess, 16.2ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)
Processed and saved: type1_subtype1_accident/ego_vehicle/Camera_Back/Town02_type001_subtype0001_scenario00013/Segmented_Data/Town02_type001_subtype0001_scenario00013_037.jpg
Saved annotations: type1_subtype1_accident/ego_vehicle/Camera_Back/Town02_type001_subtype0001_scenario00013/Annotations_JSON/Town02_type001_subtype0001_scenario00013_037.json

0: 384x640 1 car, 15.6ms
Speed: 2.7ms preprocess, 15.6m

0: 384x640 2 cars, 1 bus, 16.3ms
Speed: 2.7ms preprocess, 16.3ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)
Processed and saved: type1_subtype1_accident/ego_vehicle/Camera_Back/Town03_type001_subtype0001_scenario00024/Segmented_Data/Town03_type001_subtype0001_scenario00024_008.jpg
Saved annotations: type1_subtype1_accident/ego_vehicle/Camera_Back/Town03_type001_subtype0001_scenario00024/Annotations_JSON/Town03_type001_subtype0001_scenario00024_008.json

0: 384x640 2 cars, 1 bus, 19.9ms
Speed: 2.9ms preprocess, 19.9ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)
Processed and saved: type1_subtype1_accident/ego_vehicle/Camera_Back/Town03_type001_subtype0001_scenario00024/Segmented_Data/Town03_type001_subtype0001_scenario00024_009.jpg
Saved annotations: type1_subtype1_accident/ego_vehicle/Camera_Back/Town03_type001_subtype0001_scenario00024/Annotations_JSON/Town03_type001_subtype0001_scenario00024_009.json

0: 384x640 2 cars, 1 bus, 17.0ms
Speed: 

Speed: 3.1ms preprocess, 19.1ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)
Processed and saved: type1_subtype1_accident/ego_vehicle/Camera_Back/Town03_type001_subtype0001_scenario00024/Segmented_Data/Town03_type001_subtype0001_scenario00024_025.jpg
Saved annotations: type1_subtype1_accident/ego_vehicle/Camera_Back/Town03_type001_subtype0001_scenario00024/Annotations_JSON/Town03_type001_subtype0001_scenario00024_025.json

0: 384x640 1 car, 1 truck, 24.9ms
Speed: 3.1ms preprocess, 24.9ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)
Processed and saved: type1_subtype1_accident/ego_vehicle/Camera_Back/Town03_type001_subtype0001_scenario00024/Segmented_Data/Town03_type001_subtype0001_scenario00024_026.jpg
Saved annotations: type1_subtype1_accident/ego_vehicle/Camera_Back/Town03_type001_subtype0001_scenario00024/Annotations_JSON/Town03_type001_subtype0001_scenario00024_026.json

0: 384x640 1 car, 1 truck, 16.7ms
Speed: 4.2ms preprocess, 16.7ms infere

...

0: 384x640 1 car, 1 traffic light, 16.1ms
Speed: 2.4ms preprocess, 16.1ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)
Processed and saved: type1_subtype1_accident/ego_vehicle/Camera_BackLeft/Town10HD_type001_subtype0001_scenario00014/Segmented_Data/Town10HD_type001_subtype0001_scenario00014_055.jpg
Saved annotations: type1_subtype1_accident/ego_vehicle/Camera_BackLeft/Town10HD_type001_subtype0001_scenario00014/Annotations_JSON/Town10HD_type001_subtype0001_scenario00014_055.json

0: 384x640 1 car, 1 traffic light, 15.2ms
Speed: 2.3ms preprocess, 15.2ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)
Processed and saved: type1_subtype1_accident/ego_vehicle/Camera_BackLeft/Town10HD_type001_subtype0001_scenario00014/Segmented_Data/Town10HD_type001_subtype0001_scenario00014_056.jpg
Saved annotations: type1_subtype1_accident/ego_vehicle/Camera_BackLeft/Town10HD_type001_subtype0001_scenario00014/Annotations_JSON/Town10HD_type001_subtype0001_scenario00014_

0: 384x640 1 car, 15.4ms
Speed: 2.2ms preprocess, 15.4ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)
Processed and saved: type1_subtype1_accident/ego_vehicle/Camera_BackLeft/Town10HD_type001_subtype0001_scenario00014/Segmented_Data/Town10HD_type001_subtype0001_scenario00014_072.jpg
Saved annotations: type1_subtype1_accident/ego_vehicle/Camera_BackLeft/Town10HD_type001_subtype0001_scenario00014/Annotations_JSON/Town10HD_type001_subtype0001_scenario00014_072.json

0: 384x640 1 car, 15.4ms
Speed: 2.3ms preprocess, 15.4ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)
Processed and saved: type1_subtype1_accident/ego_vehicle/Camera_BackLeft/Town10HD_type001_subtype0001_scenario00014/Segmented_Data/Town10HD_type001_subtype0001_scenario00014_073.jpg
Saved annotations: type1_subtype1_accident/ego_vehicle/Camera_BackLeft/Town10HD_type001_subtype0001_scenario00014/Annotations_JSON/Town10HD_type001_subtype0001_scenario00014_073.json

0: 384x640 1 car, 15.2ms

0: 384x640 1 car, 15.4ms
Speed: 2.3ms preprocess, 15.4ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)
Processed and saved: type1_subtype1_accident/ego_vehicle/Camera_BackLeft/Town07_type001_subtype0001_scenario00031/Segmented_Data/Town07_type001_subtype0001_scenario00031_015.jpg
Saved annotations: type1_subtype1_accident/ego_vehicle/Camera_BackLeft/Town07_type001_subtype0001_scenario00031/Annotations_JSON/Town07_type001_subtype0001_scenario00031_015.json

0: 384x640 1 car, 16.3ms
Speed: 2.4ms preprocess, 16.3ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)
Processed and saved: type1_subtype1_accident/ego_vehicle/Camera_BackLeft/Town07_type001_subtype0001_scenario00031/Segmented_Data/Town07_type001_subtype0001_scenario00031_016.jpg
Saved annotations: type1_subtype1_accident/ego_vehicle/Camera_BackLeft/Town07_type001_subtype0001_scenario00031/Annotations_JSON/Town07_type001_subtype0001_scenario00031_016.json

0: 384x640 2 cars, 16.0ms
Speed: 2.3ms p


0: 384x640 1 car, 15.3ms
Speed: 2.3ms preprocess, 15.3ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)
Error calculating similarity: OpenCV(4.10.0) /io/opencv/modules/imgproc/src/histogram.cpp:2031: error: (-215:Assertion failed) H1.type() == H2.type() && H1.depth() == CV_32F in function 'compareHist'

Error calculating similarity: OpenCV(4.10.0) /io/opencv/modules/imgproc/src/histogram.cpp:2031: error: (-215:Assertion failed) H1.type() == H2.type() && H1.depth() == CV_32F in function 'compareHist'

Processed and saved: type1_subtype1_accident/ego_vehicle/Camera_BackLeft/Town07_type001_subtype0001_scenario00031/Segmented_Data/Town07_type001_subtype0001_scenario00031_032.jpg
Saved annotations: type1_subtype1_accident/ego_vehicle/Camera_BackLeft/Town07_type001_subtype0001_scenario00031/Annotations_JSON/Town07_type001_subtype0001_scenario00031_032.json

0: 384x640 1 car, 15.4ms
Speed: 2.4ms preprocess, 15.4ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 

0: 384x640 (no detections), 15.3ms
Speed: 2.3ms preprocess, 15.3ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)
Processed and saved: type1_subtype1_accident/ego_vehicle/Camera_BackLeft/Town07_type001_subtype0001_scenario00031/Segmented_Data/Town07_type001_subtype0001_scenario00031_048.jpg
Saved annotations: type1_subtype1_accident/ego_vehicle/Camera_BackLeft/Town07_type001_subtype0001_scenario00031/Annotations_JSON/Town07_type001_subtype0001_scenario00031_048.json

0: 384x640 (no detections), 15.5ms
Speed: 2.3ms preprocess, 15.5ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)
Processed and saved: type1_subtype1_accident/ego_vehicle/Camera_BackLeft/Town07_type001_subtype0001_scenario00031/Segmented_Data/Town07_type001_subtype0001_scenario00031_049.jpg
Saved annotations: type1_subtype1_accident/ego_vehicle/Camera_BackLeft/Town07_type001_subtype0001_scenario00031/Annotations_JSON/Town07_type001_subtype0001_scenario00031_049.json

0: 384x640 (no detec